# 04 — Validation: July 2020 net surface shortwave (GEOS `SWGNT`) vs. CERES EBAF

Compare the model's monthly-mean net downward surface shortwave radiation for
July 2020 against the CERES EBAF Ed4.2 surface product (all-sky surface net
shortwave) on a common 1° grid.

**Why the GEOS side, not `oceQsw`.** Notebook 05 showed that the ocean-side
`mit/oceQsw` stream carries only ~12% of the true surface net shortwave (smooth,
zenith-only, noon peaks of ~100 W m⁻² where `oceQnet`'s diurnal range implies
~650–750) — consistent with a sub-surface *penetrating*-SW component rather than
the surface flux its readme entry describes. The atmosphere-side GEOS net surface
shortwave (`SWGNT`-type variable) is the field that actually forces the coupled
surface and is the correct counterpart to CERES. (`oceQnet` is healthy and remains
the ocean-side net heat flux for the other notebooks.)

**Design choices.** This is a free-running coupled nature run, so the comparison is
statistical (monthly mean, zonal means, area-weighted bias/RMSE) — never instantaneous
fields, whose cloud positions cannot match observations. The model mean subsamples the
GEOS record to ~3-hourly (8 fixed local solar times per longitude — adequate diurnal
sampling; set `TARGET_HOURS = 1` for the full record at 3× the I/O; a time-averaged
`tavg_*` collection, if selected below, makes diurnal sampling a non-issue). Statistics
are restricted to the open ocean between 60°S–60°N: poleward of that, sea-ice albedo
makes the surface net SW definitions diverge between model and satellite retrieval.

**Obtaining the CERES file** (one-time, ~10 MB): order the **EBAF Ed4.2** subset —
*Surface Fluxes* → **Net Shortwave Flux, All-Sky** (`sfc_net_sw_all_mon`), covering
**July 2020** — from https://ceres.larc.nasa.gov/data/ and upload it next to this
notebook.

Reference values: global-mean all-sky surface net SW ≈ 160–165 W m⁻²; regional monthly
biases within ±10–20 W m⁻² are typical of well-performing models. Cite CERES EBAF
(Loeb et al.; Kato et al.) when publishing — [citation to be inserted].

In [ ]:
# Environment check: run on SciServer (Kraken domain, with the Poseidon DYAMOND
# ceph volume attached), or set DYAMOND_ROOT to a local subset.
# SciServer containers do not persist `pip install --user` across restarts, so
# fall back to importing directly from the repo's src/ tree if needed.
try:
    from dyamond_fluxes import dyamond_root
except ModuleNotFoundError:
    import sys
    from pathlib import Path as _P

    sys.path.insert(0, str((_P.cwd() / ".." / "src").resolve()))
    from dyamond_fluxes import dyamond_root

root = dyamond_root()
print(f"DYAMOND root: {root}")

In [ ]:
from pathlib import Path

# --- configuration -------------------------------------------------------
MONTH_START, MONTH_END = "2020-07-01", "2020-08-01"  # [start, end)
TARGET_HOURS = 3         # subsample the GEOS record to ~this spacing (1 = all output)
DLON = DLAT = 1.0        # comparison grid (matches CERES 1 deg)
CERES_FILE = Path("CERES_EBAF_Ed4.2.1_Subset_202005-202008.nc")  # adjust to your download
CERES_VAR_CANDIDATES = ["sfc_net_sw_all_mon", "sfc_net_sw_all", "sfc_net_sw"]

# Set these two from the inventory printed by the next cell (or leave None to
# auto-pick a variable whose long_name reads like "surface net ... shortwave").
SW_COLLECTION = None     # e.g. "geosgcm_surf"
SW_VAR = None            # e.g. "SWGNT"

FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)
CACHE = Path(f"sw_geos_{MONTH_START[:7]}_1deg.nc")  # gitignored (*.nc)

## Which GEOS collection holds the net surface shortwave?

Scan every collection's first file (metadata only — fast) for shortwave-related
variables, then pick. GEOS/MERRA-2 naming: `SWGNT` = surface net downward shortwave;
`SWGDN` = surface downwelling only (do **not** use — it ignores ocean albedo).

In [ ]:
from dyamond_fluxes import list_geos_collections, peek_variables

hits: dict[str, list[tuple[str, str]]] = {}
for coll in list_geos_collections():
    try:
        for var, long_name in peek_variables(coll).items():
            if "shortwave" in long_name.lower() or var.upper().startswith("SW"):
                hits.setdefault(coll, []).append((var, long_name))
    except Exception as err:  # inventory pass: report and continue
        print(f"{coll}: could not read ({err})")

for coll, pairs in hits.items():
    print(f"{coll}:")
    for var, long_name in pairs:
        print(f"    {var:16s} {long_name}")

if SW_COLLECTION is None or SW_VAR is None:
    # Prefer the canonical name, then any net-surface-shortwave long_name.
    for coll, pairs in hits.items():
        for var, long_name in pairs:
            text = long_name.lower()
            if var == "SWGNT" or ("net" in text and "surface" in text and "shortwave" in text):
                SW_COLLECTION, SW_VAR = coll, var
                break
        if SW_VAR:
            break
assert SW_COLLECTION and SW_VAR, "no candidate found - set SW_COLLECTION/SW_VAR manually"
print(f"\nusing {SW_COLLECTION}:{SW_VAR}")

In [ ]:
# Dask cluster for the parallel multi-file open and the time mean.
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit="8GB")
client = Client(cluster)
client

## Model: July-2020 mean net surface SW on the 1° grid

Time-mean on the native c1440 cubed sphere, then binning of the cell centers to 1°
(cells are quasi-uniform ~7 km, so unweighted bin means are adequate at 1°). The ocean
mask comes from the MITgcm `Depth` field binned to the same grid. Cached to NetCDF.

In [ ]:
import numpy as np
import xarray as xr

from dyamond_fluxes import (
    bin_to_latlon,
    collection_files,
    load_geos_coords,
    open_grid,
    to_positive_down,
)

if CACHE.exists():
    cache_ds = xr.open_dataset(CACHE)
    sw_model, ocean_frac = cache_ds["sw_model"], cache_ds["ocean_frac"]
    print(f"loaded cached model mean from {CACHE}")
else:
    files, times = collection_files(SW_COLLECTION, start=MONTH_START, end=MONTH_END)
    cadence_h = float(np.median(np.diff(times)) / np.timedelta64(1, "h"))
    stride = max(1, round(TARGET_HOURS / cadence_h))
    files, times = files[::stride], times[::stride]
    print(f"{len(files)} files, native cadence {cadence_h:.2f} h, stride {stride}")

    ds = xr.open_mfdataset(
        files, combine="nested", concat_dim="time", parallel=True,
        data_vars="minimal", coords="minimal", compat="override",
    )
    sw = ds[SW_VAR]
    try:
        sw = to_positive_down(sw)
    except ValueError:
        print(f"no sign metadata on {SW_VAR}; assuming GEOS radiative convention "
              "(positive down)")
    sw_mean = sw.mean("time").squeeze().load()

    coords = load_geos_coords()
    glon = next(coords[k] for k in ("lons", "lon", "longitude") if k in coords)
    glat = next(coords[k] for k in ("lats", "lat", "latitude") if k in coords)
    assert glon.size == sw_mean.size, (glon.shape, sw_mean.shape)

    sw_model = bin_to_latlon(sw_mean, glon, glat, dlon=DLON, dlat=DLAT)
    sw_model.name = "sw_model"
    sw_model.attrs.update(
        long_name=f"GEOS {SW_VAR} net downward surface shortwave, mean {MONTH_START[:7]}",
        units="W m-2",
        source=f"{SW_COLLECTION}:{SW_VAR}, ~{TARGET_HOURS}-hourly subsample",
    )

    grid = open_grid()  # ocean mask from the MITgcm bathymetry
    ocean_frac = bin_to_latlon(
        (grid.Depth > 0).astype("f4"), grid.XC, grid.YC, area=grid.rA,
        dlon=DLON, dlat=DLAT,
    )
    ocean_frac.name = "ocean_frac"

    xr.Dataset({"sw_model": sw_model, "ocean_frac": ocean_frac}).to_netcdf(CACHE)
    print(f"cached to {CACHE}")
sw_model

## CERES EBAF surface net shortwave, July 2020

CERES uses 0–360° longitudes and its own variable naming; wrap to −180–180° and align
with the model's bin centers (both grids are 1° with centers at ±0.5°).

In [ ]:
if not CERES_FILE.exists():
    raise FileNotFoundError(
        f"{CERES_FILE} not found. Order the EBAF Ed4.2 surface subset from "
        "https://ceres.larc.nasa.gov/data/ (see the notebook header) and upload it here."
    )

ceres_ds = xr.open_dataset(CERES_FILE)
var = next((v for v in CERES_VAR_CANDIDATES if v in ceres_ds), None)
if var is None:
    raise KeyError(
        f"None of {CERES_VAR_CANDIDATES} found; available: {list(ceres_ds.data_vars)}"
    )

sw_ceres = ceres_ds[var].sel(time=MONTH_START[:7]).squeeze()
sw_ceres = sw_ceres.assign_coords(lon=(((sw_ceres.lon + 180) % 360) - 180)).sortby("lon")
sw_ceres = sw_ceres.reindex_like(sw_model, method="nearest", tolerance=0.51)
print(f"CERES variable: {var}; global mean {float(sw_ceres.mean()):.1f} W m-2")

## Maps: model, CERES, and bias

In [ ]:
import matplotlib.pyplot as plt

try:
    import cmocean

    CMAP_SEQ, CMAP_DIV = cmocean.cm.thermal, cmocean.cm.balance
except ImportError:
    CMAP_SEQ, CMAP_DIV = "inferno", "RdBu_r"

ocean = (ocean_frac > 0.5) & sw_model.notnull()

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True, sharey=True)
for ax, (da, label) in zip(
    axes,
    [(sw_model.where(ocean), "GEOS-MITgcm DYAMOND"),
     (sw_ceres.where(ocean), "CERES EBAF Ed4.2")],
):
    pc = ax.pcolormesh(da.lon, da.lat, da, cmap=CMAP_SEQ, vmin=0, vmax=320)
    ax.set_title(f"{label} — net surface shortwave, July 2020")
fig.colorbar(pc, ax=axes, shrink=0.8, label="W m$^{-2}$")
fig.savefig(FIGDIR / "qsw_validation_maps.png", dpi=200, bbox_inches="tight")

In [ ]:
bias = (sw_model - sw_ceres).where(ocean)

fig, ax = plt.subplots(figsize=(11, 4.5))
pc = ax.pcolormesh(bias.lon, bias.lat, bias, cmap=CMAP_DIV, vmin=-60, vmax=60)
fig.colorbar(pc, ax=ax, label="W m$^{-2}$")
ax.set_title("Model minus CERES EBAF, net surface shortwave, July 2020")
fig.savefig(FIGDIR / "qsw_validation_bias.png", dpi=200, bbox_inches="tight")

## Zonal means and area-weighted statistics (ocean, 60°S–60°N)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sw_model.where(ocean).mean("lon").plot(y="lat", ax=ax, label="model (GEOS)")
sw_ceres.where(ocean).mean("lon").plot(y="lat", ax=ax, label="CERES EBAF")
ax.set_xlabel("W m$^{-2}$")
ax.set_ylabel("latitude")
ax.set_title("Zonal-mean net surface SW over ocean, July 2020")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(FIGDIR / "qsw_validation_zonal.png", dpi=200, bbox_inches="tight")

In [ ]:
# Area weighting: bin areas on a regular lat-lon grid scale with cos(lat).
w = np.cos(np.deg2rad(sw_model.lat)).broadcast_like(sw_model)
inband = ocean & (abs(sw_model.lat) < 60) & sw_ceres.notnull()


def wstat(da):
    return float(da.where(inband).weighted(w.where(inband, 0.0)).mean())


bias_mean = wstat(sw_model - sw_ceres)
rmse = np.sqrt(wstat((sw_model - sw_ceres) ** 2))
print(f"model ocean mean (60S-60N):  {wstat(sw_model):7.1f} W m-2")
print(f"CERES ocean mean (60S-60N):  {wstat(sw_ceres):7.1f} W m-2")
print(f"mean bias (model - CERES):   {bias_mean:7.1f} W m-2")
print(f"RMSE of 1-deg monthly bins:  {rmse:7.1f} W m-2")

## Interpretation guide

- **Mean bias within ±10 W m⁻²** and RMSE of order 15–25 W m⁻² for 1° monthly bins would
  place the simulation among well-performing coupled models for surface SW.
- **Positive bias under the stratocumulus decks** (SE Pacific, SE Atlantic, off
  California) is the classic too-few/too-thin low clouds signature.
- **A uniform offset** points to a systematic difference — aerosol/clear-sky
  transmission, ocean albedo, or diurnal sampling (an instantaneous collection at
  `TARGET_HOURS = 3` samples 8 fixed local solar times; set it to 1, or pick a
  `tavg_*` collection, to rule that out).
- The GEOS field is the flux at the air–sea interface, the same quantity CERES
  estimates — a cleaner comparison than any ocean-side diagnostic. Statistics still
  stop at 60° because of sea-ice albedo definitions.
- Cross-check with the ocean side: binned `oceQnet` minus this SW field gives the
  non-solar flux; and notebook 05 documents why `mit/oceQsw` itself (a ~12%
  penetrating-SW stream) must not be used here.

Next validation steps: repeat for a boreal-winter month (January 2021); compare the
diurnal-cycle composite against CERES SYN1deg-1H; point comparisons against the
TAO/PIRATA/RAMA buoy downwelling SW (with an albedo factor ≈ 0.94) or the OceanSITES
flux reference stations.